# Validate ConvNeXt V2-H on the 2,000-image RSNA subset — Colab

Reads everything from your Drive folder, extracts the DICOMs to the Colab runtime,
scores them with ConvNeXt and reports AUC. No Kaggle API, no `kaggle.json`, no
downloads.

### What it expects on Drive

```
MyDrive/rasna_validation/
    convnextv2h_model_bundle.pth
    model_input_v3.py
    preprocessing.py
    convnext_model.py
    rsna_subset_2000/
        parts/      rsna_2000_part00.tar … part07.tar
        metadata/   rsna_validation_2000_metadata.csv, parts_manifest.json,
                    rsna_dev_holdout_split.json
```

That is the layout you uploaded. If the folder is named differently, change `ROOT` in
section 2.

### What you need before running

**A GPU runtime.** Runtime → Change runtime type → T4. On CPU a single image takes
about 70 s, so 2,000 would take over a day.

### Rough budget

Extracting ~12 GB from Drive takes 10–20 minutes depending on Drive speed. Scoring
measured 1.0 s per image on a T4, so about 35 minutes. Call it an hour.

**Nothing on Drive is modified.** The tars are read, not moved or deleted; the DICOMs
are extracted to the Colab runtime's own disk, which is wiped when the session ends.

### What this measures, and what it does not

The subset is 800 cancer-positive and 1,200 negative images — far richer in cancer than
screening reality. AUC is rank-based so that balance does not distort it, and AUC is
what gets reported. **PPV, recall per 1,000, sensitivity at a threshold and calibration
are not reported**, because on this subset they would describe a population that does
not exist. No threshold is fitted here either.

## 1. Runtime check and dependencies

In [3]:
import subprocess, sys, json, os, platform
# kaggle 1.6.17 still works for `kernels output`, but its dataset download is broken
# against the current server (404 "No gcs url found"). kagglehub is Kaggle's supported
# library for dataset pulls and is used for that step.
subprocess.run('pip install -q timm pydicom dicomsdl pylibjpeg pylibjpeg-openjpeg '
               '"kaggle==1.6.17" kagglehub', shell=True)

import torch, numpy as np, pandas as pd, cv2
cv2.setNumThreads(1)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', DEVICE)
if DEVICE.type == 'cuda':
    print('gpu    :', torch.cuda.get_device_name(0))
else:
    print('\nNo GPU. ConvNeXt V2-H at 2048x1024 on CPU takes ~70 s per image, so 2,000')
    print('images is over a day. Runtime -> Change runtime type -> T4, then re-run.')

vers = {}
for m in ('timm', 'pydicom', 'dicomsdl', 'pylibjpeg'):
    try:
        vers[m] = getattr(__import__(m), '__version__', 'present')
    except Exception:
        vers[m] = 'absent'
print('\n' + json.dumps(vers, indent=2))
if vers['dicomsdl'] == 'absent' and vers['pylibjpeg'] == 'absent':
    raise SystemExit(
        'No JPEG2000 decoder. RSNA DICOMs are JPEG2000 and pydicom alone cannot read '
        'them, so every image would fail. Re-run this cell, or restart the runtime and '
        'try again.')

device : cuda
gpu    : Tesla T4

{
  "timm": "1.0.30",
  "pydicom": "3.0.2",
  "dicomsdl": "present",
  "pylibjpeg": "2.1.0"
}


## 2. Mount Drive and find everything

Nothing is downloaded. If a file is missing, this says which one and stops rather than
failing later with something cryptic.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/rasna_validation'      # change if yours differs
if not os.path.isdir(ROOT):
    raise SystemExit(
        '%r does not exist. Open the file pane on the left, find your folder under '
        'drive/MyDrive, and set ROOT to its path.' % ROOT)

import glob, shutil

def find(pattern):
    hits = sorted(glob.glob(os.path.join(ROOT, '**', pattern), recursive=True))
    return hits[0] if hits else None

NEEDED = {
    'bundle':    'convnextv2h_model_bundle.pth',
    'contract':  'model_input_v3.py',
    'prep':      'preprocessing.py',
    'loader':    'convnext_model.py',
    'metadata':  'rsna_validation_*_metadata.csv',
    'manifest':  'parts_manifest.json',
    'split':     'rsna_dev_holdout_split.json',
}
PATHS, missing = {}, []
for key, pat in NEEDED.items():
    p = find(pat)
    if p:
        PATHS[key] = p
        print('%-10s %-52s %8.1f MB' % (key, p.replace(ROOT + '/', ''),
                                        os.path.getsize(p) / 1e6))
    else:
        missing.append(pat)
        print('%-10s NOT FOUND (%s)' % (key, pat))

PART_FILES = sorted(glob.glob(os.path.join(ROOT, '**', 'rsna_*_part*.tar'),
                              recursive=True))
print('\ntar parts  %d  (%.2f GB)'
      % (len(PART_FILES), sum(os.path.getsize(p) for p in PART_FILES) / 1e9))
if not PART_FILES:
    missing.append('rsna_*_part*.tar')

if missing:
    raise SystemExit('missing from %s:\n%s'
                     % (ROOT, '\n'.join('  ' + m for m in missing)))
print('\neverything found on Drive  [PASS]')

# The runtime's own disk, not Drive: extracting 12 GB onto Drive would be slow and
# would eat quota for files that are only needed while this notebook runs.
IMAGES = '/content/images'
os.makedirs(IMAGES, exist_ok=True)
print('\nimages will be extracted to %s (runtime disk, wiped at session end)'
      % IMAGES)
print('free runtime disk: %.1f GB' % (shutil.disk_usage('/content').free / 1e9))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bundle     convnextv2h_model_bundle.pth                           2630.1 MB
contract   model_input_v3.py                                         0.0 MB
prep       preprocessing.py                                          0.0 MB
loader     convnext_model.py                                         0.0 MB
metadata   rsna_subset_2000/metadata/rsna_validation_2000_metadata.csv      0.2 MB
manifest   rsna_subset_2000/metadata/parts_manifest.json             0.0 MB
split      rsna_dev_holdout_split.json                               0.1 MB

tar parts  8  (11.72 GB)

everything found on Drive  [PASS]

images will be extracted to /content/images (runtime disk, wiped at session end)
free runtime disk: 207.5 GB


## 3. Extract the tar parts

Each part's SHA-256 is checked against `parts_manifest.json` before extraction, so a
file that got corrupted on the way to Drive is caught here rather than becoming a
silently short dataset.

**The tars on Drive are not touched** — they are read, and the DICOMs land on the
runtime's disk. Re-running skips parts already extracted.

In [5]:
import tarfile, hashlib

def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

MAN = json.load(open(PATHS['manifest']))
print('manifest: %d parts, %d files' % (len(MAN), sum(e['files'] for e in MAN)))

by_name = {os.path.basename(p): p for p in PART_FILES}
done_marker = os.path.join(IMAGES, '.extracted')
already = set()
if os.path.exists(done_marker):
    already = set(open(done_marker).read().split())
    print('already extracted: %d parts' % len(already))

import time
t0 = time.perf_counter()
for e in MAN:
    if e['part'] in already:
        print('  %-26s skipped (done)' % e['part'])
        continue
    src = by_name.get(e['part'])
    if not src:
        raise SystemExit('%s is in the manifest but not on Drive' % e['part'])
    if os.path.getsize(src) != e['bytes']:
        raise SystemExit('%s is %d bytes, manifest says %d - re-upload this part'
                         % (e['part'], os.path.getsize(src), e['bytes']))
    got = sha256_file(src)
    if got != e['sha256']:
        raise SystemExit('%s failed its hash check (%s vs %s) - it arrived corrupt, '
                         're-upload this part' % (e['part'], got[:16], e['sha256'][:16]))
    with tarfile.open(src) as tf:
        tf.extractall(IMAGES)
    already.add(e['part'])
    open(done_marker, 'w').write('\n'.join(sorted(already)))
    print('  %-26s OK  %d files  (%.1f min elapsed)'
          % (e['part'], e['files'], (time.perf_counter() - t0) / 60), flush=True)

n_dcm = sum(1 for _ in glob.iglob(IMAGES + '/**/*.dcm', recursive=True))
print('\nDICOMs extracted: %d' % n_dcm)
print('free runtime disk: %.1f GB' % (shutil.disk_usage('/content').free / 1e9))

manifest: 8 parts, 2000 files


/tmp/ipykernel_924/461060237.py:37: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(IMAGES)


  rsna_2000_part00.tar       OK  239 files  (0.6 min elapsed)
  rsna_2000_part01.tar       OK  270 files  (1.0 min elapsed)
  rsna_2000_part02.tar       OK  243 files  (1.5 min elapsed)
  rsna_2000_part03.tar       OK  252 files  (2.0 min elapsed)
  rsna_2000_part04.tar       OK  254 files  (2.5 min elapsed)
  rsna_2000_part05.tar       OK  266 files  (3.0 min elapsed)
  rsna_2000_part06.tar       OK  261 files  (3.5 min elapsed)
  rsna_2000_part07.tar       OK  215 files  (4.1 min elapsed)

DICOMs extracted: 2000
free runtime disk: 184.0 GB


## 4. Labels, and the holdout gate

Labels come from the metadata CSV the packaging notebook wrote. The split file came
along in the kernel output, so the holdout check still runs here rather than being
taken on trust.

In [6]:
EVAL = pd.read_csv(PATHS['metadata'])
EVAL['patient_id'] = EVAL['patient_id'].astype(str)
EVAL['image_id'] = EVAL['image_id'].astype(str)
EVAL['image_key'] = EVAL['patient_id'] + '_' + EVAL['image_id']
EVAL['local_path'] = [os.path.join(IMAGES, 'images', p, '%s.dcm' % i)
                      for p, i in zip(EVAL['patient_id'], EVAL['image_id'])]
print('metadata rows : %d' % len(EVAL))
print('positive      : %d' % int((EVAL['cancer'] == 1).sum()))
print('negative      : %d' % int((EVAL['cancer'] == 0).sum()))
print('patients      : %d' % EVAL['patient_id'].nunique())

if PATHS.get('split'):
    SP = json.load(open(PATHS['split']))
    DEV = {str(p) for p in SP.get('development', [])}
    HOLD = {str(p) for p in SP.get('holdout', [])}
    print('\ndevelopment %d / holdout %d' % (len(DEV), len(HOLD)))
    leak = set(EVAL['patient_id']) & HOLD
    outside = set(EVAL['patient_id']) - DEV
    print('holdout patients in this set : %d %s'
          % (len(leak), '[PASS]' if not leak else ''))
    if leak:
        raise SystemExit('HOLDOUT LEAK: %d patients' % len(leak))
    if outside:
        raise SystemExit('%d patients are not in the development list' % len(outside))
else:
    raise SystemExit('rsna_dev_holdout_split.json not on Drive; without it '
                     'the holdout cannot be re-checked here, and taking it on trust is '
                     'the one thing sealing it was meant to prevent.')

missing = [p for p in EVAL['local_path'] if not os.path.exists(p)]
print('\nfiles on disk : %d of %d' % (len(EVAL) - len(missing), len(EVAL)))
if missing:
    raise SystemExit('%d DICOMs are missing, e.g. %r. Re-run section 3.'
                     % (len(missing), missing[:2]))
print('every selected DICOM is present  [PASS]')

metadata rows : 2000
positive      : 800
negative      : 1200
patients      : 1423

development 6214 / holdout 2663
holdout patients in this set : 0 [PASS]

files on disk : 2000 of 2000
every selected DICOM is present  [PASS]


## 5. The preprocessing module and the model

`preprocessing.py` must be the fixed build — `322c762c…` was missing `import math`,
which only shows up on images wider than 1024 px, i.e. every real mammogram. The hash
is checked here, and if the dataset copy is stale you can upload the corrected file
instead of going back to Kaggle.

In [7]:
import importlib.util

PREP_SHA_OK = '9990782d3f0edace087b3bf3499741dc7dc0bbfbd3ca2ddd5a958cfcbbbe2acd'

prep_path = None
for p in sorted(glob.glob(os.path.join(ROOT, '**', 'preprocessing.py'), recursive=True)
                + glob.glob('/content/preprocessing.py')):
    if sha256_file(p) == PREP_SHA_OK:
        prep_path = p
        break

if prep_path is None:
    print('The preprocessing.py on Drive is not the fixed build (expected sha %s).'
          % PREP_SHA_OK[:12])
    print('Upload preprocessing_FIXED.py, the one delivered with this notebook.')
    from google.colab import files
    up = files.upload()
    name = next((k for k in up if k.endswith('.py')), None)
    if not name:
        raise SystemExit('nothing uploaded')
    open('/content/preprocessing.py', 'wb').write(up[name])
    if sha256_file('/content/preprocessing.py') != PREP_SHA_OK:
        raise SystemExit('the uploaded file is not the expected build')
    prep_path = '/content/preprocessing.py'
print('prep module :', prep_path)

CONTRACT_PATH, CM_PATH, BUNDLE = PATHS['contract'], PATHS['loader'], PATHS['bundle']
for n, v in (('model_input_v3.py', CONTRACT_PATH), ('convnext_model.py', CM_PATH),
             ('convnextv2h_model_bundle.pth', BUNDLE)):
    print('%-28s %s' % (n, v.replace(ROOT + '/', '')))

spec = importlib.util.spec_from_file_location('pinned_prep', prep_path)
PREPMOD = importlib.util.module_from_spec(spec)
sys.modules['pinned_prep'] = PREPMOD
spec.loader.exec_module(PREPMOD)
_c = dict(PREPMOD.CONTRACT)
if (_c['target_h'], _c['target_w']) != (2048, 1024) or _c['clahe'] is not False:
    raise SystemExit('prep contract is %dx%d clahe=%r, expected 2048x1024 clahe=False'
                     % (_c['target_h'], _c['target_w'], _c['clahe']))
PREP = {'name': 'convnextv2h', 'module': PREPMOD, 'target_h': 2048, 'target_w': 1024}
print('\nprep contract: %s %dx%d clahe=%s'
      % (_c['prep_version'], _c['target_h'], _c['target_w'], _c['clahe']))

sys.path.insert(0, os.path.dirname(CM_PATH))
spec = importlib.util.spec_from_file_location('convnext_model', CM_PATH)
convnext_model = importlib.util.module_from_spec(spec)
sys.modules['convnext_model'] = convnext_model
spec.loader.exec_module(convnext_model)

MODEL_NET, CONTRACT_MOD, META = convnext_model.load_bundle(BUNDLE, CONTRACT_PATH,
                                                           device=str(DEVICE))
MODEL_NET.eval()
CLASS_NAMES = ('Normal', 'Benign', 'Malignant')
if list(META['class_names']) != list(CLASS_NAMES):
    raise SystemExit('bundle class order %r' % (META['class_names'],))
TEMPERATURE = float(META['temperature'])
if not TEMPERATURE > 0:
    raise SystemExit('bundle temperature %r' % TEMPERATURE)
print('\nformat      %s' % META['format'])
print('encoder     %s' % META['encoder'])
print('parameters  %s' % format(META['parameters'], ','))
print('input       %dx%d' % (META['model_h'], META['model_w']))
print('temperature %.12f  (from the bundle, calibrated=%s)'
      % (TEMPERATURE, META['calibrated']))
print('verify_bundle passed')

prep module : /content/drive/MyDrive/rasna_validation/preprocessing.py
model_input_v3.py            model_input_v3.py
convnext_model.py            convnext_model.py
convnextv2h_model_bundle.pth convnextv2h_model_bundle.pth

prep contract: prep-v2 2048x1024 clahe=False

format      convnextv2h-classical-bundle/1
encoder     convnextv2_huge.fcmae_ft_in22k_in1k_512
parameters  657,481,091
input       2048x1024
temperature 2.783446788788  (from the bundle, calibrated=True)
verify_bundle passed


## 6. Stage A and Stage B, then a self-test

These two functions are copied verbatim from your scoring notebook. The self-test runs
them on synthetic frames including RSNA-sized ones, because the crop detector takes a
different path above 1024 px — that branch is where a missing import hid before.

In [8]:
import math


class StageAError(RuntimeError):
    """A file could not be decoded into something Stage B can consume."""


def _dicom_dicomsdl(path):
    import dicomsdl
    d = dicomsdl.open(path)
    a = d.pixelData(storedvalue=True)
    info = d.getPixelDataInfo() or {}
    return a, {'photometric': str(info.get('PhotometricInterpretation', '') or ''),
               'bits_stored': info.get('BitsStored'), 'decoder': 'dicomsdl'}


def _dicom_pydicom(path):
    import pydicom
    d = pydicom.dcmread(path)
    return d.pixel_array, {
        'photometric': str(getattr(d, 'PhotometricInterpretation', '') or ''),
        'bits_stored': getattr(d, 'BitsStored', None), 'decoder': 'pydicom'}


def stage_a_decode(path):
    """DICOM or PNG/JPEG -> (integer grayscale array, meta). No windowing/crop/resize."""
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.dcm', '.dicom', ''):
        err, a = None, None
        for fn in (_dicom_dicomsdl, _dicom_pydicom):
            try:
                a, meta = fn(path)
                break
            except Exception as exc:
                err = exc
        if a is None:
            raise StageAError('could not decode %r with either decoder: %r'
                              % (path, err))
        a = np.asarray(a)
        if a.ndim == 3:
            a = a[..., 0]
        if a.ndim != 2:
            raise StageAError('%r decoded to %r, expected 2-D' % (path, a.shape))
        bits = meta.get('bits_stored')
        try:
            bits = int(bits)
        except (TypeError, ValueError):
            bits = None
        if a.dtype.kind == 'f':
            a = np.rint(a)
        a = np.clip(a, 0, None)
        maxval = (1 << bits) - 1 if bits and 1 <= bits <= 16 else int(a.max())
        maxval = max(int(maxval), int(a.max()), 1)
        a = a.astype(np.uint16) if maxval > 255 else a.astype(np.uint8)
        inverted = meta['photometric'].upper() == 'MONOCHROME1'
        if inverted:
            a = (np.full_like(a, maxval) - a).astype(a.dtype)
        meta.update(inverted=inverted, bits_stored_used=bits, maxval=int(maxval),
                    h=int(a.shape[0]), w=int(a.shape[1]), dtype=str(a.dtype),
                    windowed=False, cropped=False, resized=False)
        return a, meta

    a = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if a is None:
        raise StageAError('could not read %r' % path)
    if a.ndim == 3:
        planes = [a[:, :, i] for i in range(min(3, a.shape[2]))]
        if not all(np.array_equal(planes[0], p) for p in planes[1:]):
            raise StageAError('%r is genuinely coloured; a mammogram is grayscale'
                              % path)
        a = planes[0]
    return a, {'photometric': 'n/a', 'decoder': 'opencv', 'inverted': False,
               'bits_stored_used': None, 'maxval': int(a.max()),
               'h': int(a.shape[0]), 'w': int(a.shape[1]), 'dtype': str(a.dtype),
               'windowed': False, 'cropped': False, 'resized': False}


def prepare_array(prep, arr):
    """Run one model's Stage B on a decoded array. Mirrors prepare_raw exactly.

    Returns (float32 tensor (3,H,W), geometry). The geometry carries the same keys
    prepare_raw records, so attribution mapping and the manifests are unchanged.
    """
    m = prep['module']
    info = m.crop_box(arr)
    y0, y1, x0, x1 = info['box']
    crop = arr[y0:y1, x0:x1]
    img8, conv = m.to_8bit(crop)
    padded, geom = m.fit_and_pad(img8)
    geom.update(entry='raw', orig_h=int(arr.shape[0]), orig_w=int(arr.shape[1]),
                crop_y0=y0, crop_y1=y1, crop_x0=x0, crop_x1=x1,
                crop_h=int(crop.shape[0]), crop_w=int(crop.shape[1]),
                crop_status=info['status'], crop_reason=info['reason'],
                to8=conv, padded_8bit=padded)
    H, W = prep['target_h'], prep['target_w']
    if padded.shape != (H, W):
        raise RuntimeError('%r: Stage B produced %r, contract says %r'
                           % (prep['name'], padded.shape, (H, W)))
    return m.normalise(padded), geom


def _synth(h, w, bits, seed=0):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:h, 0:w]
    blob = (((yy - h * .52) / (h * .40)) ** 2
            + ((xx - w * .18) / (w * .62)) ** 2) <= 1.0
    img = np.zeros((h, w), np.float32)
    img[blob] = 0.45 + 0.35 * rng.random(int(blob.sum()))
    img[12:40, w - 90:w - 20] = 0.98
    mx = (1 << bits) - 1
    return (np.clip(img, 0, 1) * mx).astype(np.uint16 if mx > 255 else np.uint8)

bad = []
for bits, h, w in ((8, 880, 620), (12, 880, 620), (16, 880, 620),
                   (12, 2776, 2082), (16, 3328, 2560)):
    x, g = prepare_array(PREP, _synth(h, w, bits, bits))
    ok = (g['padded_8bit'].shape == (2048, 1024)
          and g['padded_8bit'].dtype == np.uint8
          and x.shape == (3, 2048, 1024) and g['entry'] == 'raw')
    print('%2d-bit %4dx%-4d  container %-6s scale %.3f  %s'
          % (bits, h, w, g['to8'].get('container'), g['scale'],
             'PASS' if ok else 'FAIL'))
    if not ok:
        bad.append((bits, h, w))
if bad:
    raise SystemExit('Stage B self-test failed at %r' % bad)
print('\nStage B self-test passed')

 8-bit  880x620   container 255    scale 2.004  PASS
12-bit  880x620   container 4095   scale 2.004  PASS
16-bit  880x620   container 65535  scale 2.004  PASS
12-bit 2776x2082  container 4095   scale 0.596  PASS
16-bit 3328x2560  container 65535  scale 0.485  PASS

Stage B self-test passed


## 7. Score

One image at a time under `torch.inference_mode()`. Only the probability row is kept;
the tensor is freed after each forward pass. Rows flush to
`predictions_partial.csv` every 25 images, so a disconnect costs almost nothing — re-run
this cell and it resumes.

A 16-image probe runs first and **stops if any of them fails**, so a broken pipeline
costs seconds rather than half an hour.

In [9]:
from torch.utils.data import Dataset, DataLoader

OUT_DIR = '/content/out'
os.makedirs(OUT_DIR, exist_ok=True)
PARTIAL = OUT_DIR + '/predictions_partial.csv'
FINAL = OUT_DIR + '/predictions.csv'
AMP = DEVICE.type == 'cuda'


class DicomDS(Dataset):
    def __init__(self, frame):
        self.df = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:
            arr, _ = stage_a_decode(r['local_path'])
            x, g = prepare_array(PREP, arr)
            return (torch.from_numpy(np.ascontiguousarray(x)), int(i), True,
                    str(g['crop_status']), float(g['scale']),
                    int(g['orig_h']), int(g['orig_w']))
        except Exception as exc:
            return (torch.zeros(3, 2048, 1024, dtype=torch.float32), int(i), False,
                    ('failed:%s: %s' % (type(exc).__name__, exc))[:200],
                    float('nan'), -1, -1)


ds = DicomDS(EVAL)
print('probing 16 images...')
items = [ds[i] for i in range(16)]
fails = [(int(i), cs) for _x, i, ok, cs, _s, _h, _w in items if not ok]
if fails:
    for i, cs in fails[:5]:
        print('  FAILED %s\n         %s' % (EVAL.iloc[i]['local_path'], cs))
    raise SystemExit('%d of 16 probe images failed. Fix this before the full run.'
                     % len(fails))
print('all 16 preprocessed  [PASS]')
del items

done, rows = set(), []
if os.path.exists(PARTIAL):
    prev = pd.read_csv(PARTIAL)
    rows = prev.to_dict('records')
    done = set(prev['image_key'].astype(str))
    print('resuming: %d already scored' % len(done))
TODO = EVAL[~EVAL['image_key'].isin(done)].reset_index(drop=True)
print('to score: %d of %d' % (len(TODO), len(EVAL)))

if len(TODO):
    loader = DataLoader(DicomDS(TODO), batch_size=1, shuffle=False, num_workers=2,
                        pin_memory=AMP)
    import time
    t0, seen = time.perf_counter(), 0
    with torch.inference_mode():
        for x, idx, ok, cs, sc, oh, ow in loader:
            x = x.to(DEVICE, non_blocking=True)
            if AMP:
                with torch.autocast('cuda', dtype=torch.float16):
                    logits = MODEL_NET(x)
            else:
                logits = MODEL_NET(x)
            logits = logits.float().cpu()
            del x
            probs = torch.softmax(logits / TEMPERATURE, dim=1).numpy()
            for j in range(logits.shape[0]):
                r = TODO.iloc[int(idx[j])]
                rows.append({
                    'patient_id': r['patient_id'], 'image_id': r['image_id'],
                    'image_key': r['image_key'],
                    'laterality': r.get('laterality'), 'view': r.get('view'),
                    'site_id': r.get('site_id'), 'machine_id': r.get('machine_id'),
                    'age': r.get('age'), 'cancer': int(r['cancer']),
                    'ok': bool(ok[j]), 'crop_status': str(cs[j]),
                    'scale': float(sc[j]), 'orig_h': int(oh[j]), 'orig_w': int(ow[j]),
                    'pred_class': CLASS_NAMES[int(np.argmax(probs[j]))],
                    'p_Normal': float(probs[j, 0]),
                    'p_Benign': float(probs[j, 1]),
                    'p_Malignant': float(probs[j, 2])})
            del logits, probs
            seen += 1
            if seen % 25 == 0:
                pd.DataFrame(rows).to_csv(PARTIAL, index=False)
                el = time.perf_counter() - t0
                print('  %d/%d  %.1f min  %.2f s/img  eta %.1f min'
                      % (seen, len(TODO), el / 60, el / seen,
                         (len(TODO) - seen) * el / seen / 60), flush=True)
            if seen >= 50 and not any(q['ok'] for q in rows):
                pd.DataFrame(rows).to_csv(PARTIAL, index=False)
                raise SystemExit('the first %d images all failed (%r) - stopping'
                                 % (seen, rows[-1]['crop_status']))
    pd.DataFrame(rows).to_csv(PARTIAL, index=False)
    print('\nscored %d in %.2f h' % (seen, (time.perf_counter() - t0) / 3600))

PRED = pd.DataFrame(rows)
PRED.to_csv(FINAL, index=False)
print('rows %d, failed %d' % (len(PRED), int((~PRED['ok']).sum())))

probing 16 images...
all 16 preprocessed  [PASS]
to score: 2000 of 2000
  25/2000  0.5 min  1.14 s/img  eta 37.4 min
  50/2000  0.9 min  1.06 s/img  eta 34.3 min
  75/2000  1.4 min  1.14 s/img  eta 36.6 min
  100/2000  2.0 min  1.20 s/img  eta 38.1 min
  125/2000  2.6 min  1.24 s/img  eta 38.7 min
  150/2000  3.0 min  1.22 s/img  eta 37.6 min
  175/2000  3.6 min  1.24 s/img  eta 37.6 min
  200/2000  4.1 min  1.24 s/img  eta 37.2 min
  225/2000  4.6 min  1.23 s/img  eta 36.3 min
  250/2000  5.0 min  1.21 s/img  eta 35.3 min
  275/2000  5.7 min  1.23 s/img  eta 35.5 min
  300/2000  6.2 min  1.23 s/img  eta 35.0 min
  325/2000  6.7 min  1.23 s/img  eta 34.4 min
  350/2000  7.2 min  1.23 s/img  eta 33.9 min
  375/2000  7.7 min  1.23 s/img  eta 33.4 min
  400/2000  8.2 min  1.23 s/img  eta 32.8 min
  425/2000  8.7 min  1.23 s/img  eta 32.3 min
  450/2000  9.3 min  1.24 s/img  eta 32.0 min
  475/2000  9.9 min  1.25 s/img  eta 31.8 min
  500/2000  10.4 min  1.25 s/img  eta 31.3 min
  525/2000

## 8. AUC

`patient_bootstrap_auc` and `breast_level` are verbatim from your scoring notebook, so
the aggregation rule is the project's existing one: max malignant probability per breast.
The bootstrap resamples **patients**, not images, because two views of one breast are not
independent.

In [10]:
from sklearn.metrics import roc_auc_score


def patient_bootstrap_auc(y, score, patients, n=2000, seed=0):
    """Resample PATIENTS with replacement. Returns (auc, lo, hi)."""
    y = np.asarray(y, float)
    s = np.asarray(score, float)
    p = np.asarray(patients)
    keep = np.isfinite(s) & np.isfinite(y)
    y, s, p = y[keep], s[keep], p[keep]
    if len(np.unique(y)) < 2:
        return float('nan'), float('nan'), float('nan')
    point = float(roc_auc_score(y, s))

    uniq = np.unique(p)
    by = {u: np.flatnonzero(p == u) for u in uniq}
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([by[u] for u in pick])
        yy = y[idx]
        if len(np.unique(yy)) < 2:
            continue
        out.append(roc_auc_score(yy, s[idx]))
    if not out:
        return point, float('nan'), float('nan')
    lo, hi = np.percentile(out, [2.5, 97.5])
    return point, float(lo), float(hi)


def breast_level(df, score_col='p_Malignant'):
    """Max malignant probability per breast, which is how a screening read works."""
    need = {'patient_id', 'laterality', 'cancer'}
    if not need.issubset(df.columns):
        return None
    g = (df[df['ok']]
         .groupby(['patient_id', 'laterality'])
         .agg(score=(score_col, 'max'), cancer=('cancer', 'max'))
         .reset_index())
    return g


D = PRED[PRED['ok']].copy()
if len(D) == 0:
    print(PRED['crop_status'].value_counts().to_string())
    raise SystemExit('nothing scored successfully - no result is written, because a '
                     'file of NaNs looks like an answer')
if D['cancer'].nunique() < 2:
    raise SystemExit('only one class survived; AUC is undefined')

print('scored OK       %d images, %d patients'
      % (len(D), D['patient_id'].nunique()))
print('positive        %d images' % int((D['cancer'] == 1).sum()))
print('failed          %d' % int((~PRED['ok']).sum()))
print('mean scale      %.4f   upscaled (>1): %.1f%%'
      % (D['scale'].mean(), 100.0 * (D['scale'] > 1).mean()))

img = patient_bootstrap_auc(D['cancer'], D['p_Malignant'], D['patient_id'],
                            n=2000, seed=1)
print('\nIMAGE-level  AUC %.4f   95%% CI [%.4f, %.4f]   n=%d'
      % (img[0], img[1], img[2], len(D)))

B = breast_level(D, 'p_Malignant')
brs = (float('nan'),) * 3
if B is not None and len(B):
    brs = patient_bootstrap_auc(B['cancer'], B['score'], B['patient_id'],
                                n=2000, seed=2)
    print('BREAST-level AUC %.4f   95%% CI [%.4f, %.4f]   n=%d breasts (%d positive)'
          % (brs[0], brs[1], brs[2], len(B), int((B['cancer'] == 1).sum())))
    print('  aggregation: max p_Malignant per (patient_id, laterality)')

for col in ('site_id', 'machine_id'):
    if col in D.columns and D[col].nunique() > 1:
        print('\nAUC by %s' % col)
        for v, g in D.groupby(col):
            if g['cancer'].nunique() < 2:
                print('  %-8s n=%-5d (one class only)' % (v, len(g)))
            else:
                print('  %-8s n=%-5d AUC %.4f'
                      % (v, len(g), roc_auc_score(g['cancer'], g['p_Malignant'])))

summary = {
    'label': 'RSNA development subset validation / smoke external validation',
    'environment': 'colab', 'partition': 'development', 'holdout_opened': False,
    'n_images': int(len(D)), 'n_failed': int((~PRED['ok']).sum()),
    'n_patients': int(D['patient_id'].nunique()),
    'n_positive_images': int((D['cancer'] == 1).sum()),
    'image_auc': img[0], 'image_auc_lo': img[1], 'image_auc_hi': img[2],
    'breast_auc': brs[0], 'breast_auc_lo': brs[1], 'breast_auc_hi': brs[2],
    'breast_aggregation': 'max p_Malignant per (patient_id, laterality)',
    'temperature': TEMPERATURE,
    'prep_module_sha256': PREP_SHA_OK,
    'prep_module_is_recorded_provenance_file': False,
    'metrics_withheld': ['PPV', 'calibration', 'recall_per_1000', 'sensitivity',
                         'specificity'],
    'metrics_withheld_reason': 'enriched subset (800 positive / 1200 negative); '
                               'prevalence-dependent metrics would not describe '
                               'screening, and no threshold was fitted here',
}
json.dump(summary, open(OUT_DIR + '/summary.json', 'w'), indent=2)
print('\nwrote %s/summary.json and predictions.csv' % OUT_DIR)
print('\nDownload them now - the Colab runtime is wiped when the session ends.')
try:
    from google.colab import files
    files.download(FINAL)
    files.download(OUT_DIR + '/summary.json')
except Exception as exc:
    print('(automatic download unavailable: %s - grab them from the file pane)' % exc)

scored OK       2000 images, 1423 patients
positive        800 images
failed          0
mean scale      0.5251   upscaled (>1): 0.9%

IMAGE-level  AUC 0.6430   95% CI [0.6103, 0.6757]   n=2000
BREAST-level AUC 0.7539   95% CI [0.7219, 0.7824]   n=1503 breasts (343 positive)
  aggregation: max p_Malignant per (patient_id, laterality)

AUC by site_id
  1        n=1169  AUC 0.6306
  2        n=831   AUC 0.6663

AUC by machine_id
  21       n=264   AUC 0.7451
  29       n=281   AUC 0.5544
  48       n=286   AUC 0.7251
  49       n=976   AUC 0.6547
  93       n=63    AUC 0.8741
  170      n=41    AUC 0.5949
  190      n=8     AUC 1.0000
  197      n=1     (one class only)
  210      n=21    (one class only)
  216      n=59    AUC 0.3591

wrote /content/out/summary.json and predictions.csv

Download them now - the Colab runtime is wiped when the session ends.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Reading the result

Report it as **"RSNA development subset validation / smoke external validation"**.

- the sealed holdout was not opened, and section 5 re-checked that rather than trusting it
- AUC is the headline; the subset is enriched for cancer, so PPV, recall per 1,000 and
  calibration are absent on purpose
- **not comparable to the Sprint 5 v12 operating point**, which was 12:1 with site
  weights. AUC against AUC is fine; operating point against operating point is not
- the preprocessing module is the assembled prep-v2 deployment module, not the file whose
  hash the ConvNeXt bundle recorded — `summary.json` records that explicitly

If `mean scale` comes back above 1.0 for a large share of images, the model is being fed
upscaled frames, and that belongs in the write-up next to the AUC.